# Deep Dive: Routing

## 📋 Summary

**Routing** means classifying an incoming request and **sending it to the specialist agent best suited to handle it**.
A chain runs every step for every input; a router runs **only the step that fits**.

```
                ┌──▶ Flights Agent
request ──▶ ROUTER ──▶ Visa Agent
                └──▶ Local Tips Agent
```

Why not one big agent that does everything? Specialists have **short, sharp prompts** (better quality, easier to test), and you can add a new
specialist without touching the others. The price is one extra decision: *who gets this request?*

| Router type | How it decides | Strength | Weakness |
|---|---|---|---|
| **LLM router** | An LLM reads the agent descriptions and picks one | Understands nuance and phrasing | One extra LLM call, can answer with an unexpected name |
| **Embedding router** | Cosine similarity between the request and each description | Fast, cheap, deterministic, easy to log scores | Only as good as the descriptions, no reasoning |
| **Rule / keyword router** | `if "visa" in query` | Free and predictable | Breaks on any phrasing you did not foresee |

### Key concepts in this notebook
| # | Concept | Complexity |
|---|---------|-----------|
| 1 | Specialist agents with descriptions | 🟢 Beginner |
| 2 | A basic LLM router | 🟢 Beginner |
| 3 | A robust router: JSON, validation, fallback | 🟡 Intermediate |
| 4 | Descriptions are the router's API | 🟡 Intermediate |
| 5 | Routing to agents that need prerequisites | 🟡 Intermediate |
| 6 | Embedding-based routing (cosine similarity) | 🔴 Advanced |
| 7 | Exercises | 🏋️ Practice |

**How to use this notebook:** run the cells in order (each section reuses earlier definitions). Cells marked `# TODO` are for you to fill in.
The notebook calls the OpenAI chat and embeddings APIs with cheap models.

## 0. Setup

In [ ]:
# Key input: in Colab, add OPENAI_API_KEY in the 🔑 Secrets panel (left sidebar).
import os

try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
except ImportError:
    from getpass import getpass
    if not os.environ.get("OPENAI_API_KEY"):
        os.environ["OPENAI_API_KEY"] = getpass("OPENAI_API_KEY: ")

In [ ]:
import json

import numpy as np
from openai import OpenAI
from IPython.display import Markdown, display

client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])
MODEL = "gpt-4.1-nano"                    # cheap chat model
EMBED_MODEL = "text-embedding-3-small"    # cheap embedding model


def call_llm(system_prompt: str, user_prompt: str, temperature: float = 0.0) -> str:
    resp = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        temperature=temperature,
    )
    return resp.choices[0].message.content.strip()


def call_llm_json(system_prompt: str, user_prompt: str) -> dict:
    """Forces a JSON object back (the prompt must mention JSON)."""
    resp = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        temperature=0,
        response_format={"type": "json_object"},
    )
    return json.loads(resp.choices[0].message.content)


def show(label: str, text: str):
    display(Markdown(f"**{label}**\n\n{text}\n\n---"))

## 1. Specialist Agents with Descriptions 🟢

Every agent the router can choose from needs three things:
- a **name** (the router's answer must match it),
- a **description** (what the router reads to decide, *this is the most important text in the whole system*),
- a **`run()`** that does the work.

**Running example:** a **travel concierge** with specialists for flights, visas and local tips.

In [ ]:
class SpecialistAgent:
    """A specialist: name + description (for the router) + system prompt (for the work)."""

    def __init__(self, name: str, description: str, system_prompt: str):
        self.name = name
        self.description = description
        self.system_prompt = system_prompt

    def get_description(self) -> str:
        return self.description

    def run(self, task: str, context: dict | None = None) -> str:
        user = task
        if context:  # results from other agents, if any
            extra = "\n\n".join(f"[{k}]\n{v}" for k, v in context.items())
            user = f"{task}\n\nUse this information:\n{extra}"
        return call_llm(self.system_prompt, user)


SPECS = [
    ("Flights Agent",
     "Use for questions about flights, airlines, layovers, baggage allowance and the best time to buy tickets.",
     "You are a flights specialist. Answer in at most 80 words with concrete, practical advice."),
    ("Visa Agent",
     "Use for passports, visas, entry requirements, vaccination certificates and border-crossing documents.",
     "You are a visa and entry-requirements specialist. Answer in at most 80 words and remind the user to verify with the official government site."),
    ("Local Tips Agent",
     "Use for things to do, food, neighbourhoods, local customs, tipping etiquette and weather at the destination.",
     "You are a local-guide specialist. Answer in at most 80 words with specific suggestions."),
]


def build_agents(vague: bool = False) -> list:
    """vague=True gives every agent the same useless description (used in section 4)."""
    return [SpecialistAgent(n, "Helps with travel questions." if vague else d, s) for n, d, s in SPECS]


agents = build_agents()
for a in agents:
    print(f"- {a.name}: {a.get_description()}")

## 2. A Basic LLM Router 🟢

The router builds its prompt **dynamically from the agents' descriptions**, so adding an agent to the list is enough to make it routable.
It asks for the agent's **exact name**, then looks that agent up and runs it.

In [ ]:
def basic_router(task: str, agents: list) -> str:
    options = "\n".join(f"- {a.name}: {a.get_description()}" for a in agents)
    system = f"""You are a routing assistant. Pick the best agent for the user's task.
Available agents:
{options}

Respond ONLY with the exact agent name, nothing else."""
    choice = call_llm(system, task)
    for a in agents:
        if a.name == choice:
            print(f"➡️  Routed to: {a.name}")
            return a.run(task)
    return f"Error: no agent named {choice!r}"


show("Answer", basic_router("Do I need a visa to visit Japan with a Brazilian passport?", agents))
show("Answer", basic_router("Which neighbourhood in Lisbon has the best pastel de nata?", agents))

### 🧐 Observation
- The router returned a *name*, and we matched it with `==`. What happens if the model answers `"Visa Agent."` (with a period) or `"the Visa Agent"`?
- What happens with a request that no agent covers ("Write me a poem")? What would you want to happen instead of an error string?

## 3. A Robust Router: JSON, Validation, Fallback 🟡

Three upgrades make routing production-friendly:

1. **Structured output**: ask for JSON with the agent name *and a short reason* (useful for logs).
2. **Validation**: check that the name is really one of the options.
3. **Fallback**: a default agent (or a polite "I can't help") for anything unrecognised.

In [ ]:
FALLBACK_NAME = "NONE"


def llm_route(task: str, agents: list) -> dict:
    """Returns {"agent": <valid name or NONE>, "reason": str}."""
    names = [a.name for a in agents]
    options = "\n".join(f"- {a.name}: {a.get_description()}" for a in agents)
    system = f"""You are a routing assistant. Choose the single best agent for the user's task.
Available agents:
{options}

Return JSON exactly like {{"agent": "<one of {names} or {FALLBACK_NAME}>", "reason": "<one short sentence>"}}.
Use {FALLBACK_NAME} if no agent is a good fit."""
    decision = call_llm_json(system, task)
    if decision.get("agent") not in names:
        decision = {"agent": FALLBACK_NAME, "reason": decision.get("reason", "invalid agent name")}
    return decision


def route_and_run(task: str, agents: list) -> str:
    decision = llm_route(task, agents)
    print(f"➡️  {decision['agent']}: {decision['reason']}")
    if decision["agent"] == FALLBACK_NAME:
        return "Sorry, I can only help with flights, visas and local tips."
    agent = next(a for a in agents if a.name == decision["agent"])
    return agent.run(task)


for q in [
    "How much checked baggage can I take on a Lufthansa economy ticket?",
    "Write me a poem about autumn.",
    "Do I need a visa for Canada and what should I eat in Montreal?",  # two intents
]:
    print("\nUser:", q)
    print(route_and_run(q, agents))

### 🧐 Observation
- The third request has **two intents** (visa + food). The router picked one. What are two ways to handle multi-intent requests? (Preview: **parallelization** and **orchestrator-workers** are coming in the next notebooks.)
- The `reason` field costs a few tokens. Where would you use it?

## 4. Descriptions Are the Router's API 🟡

The router never sees your agents' code, only their **descriptions**. Vague descriptions produce vague routing.
Let's **measure** it with a small labeled test set: how often does the router pick the expected agent with useless descriptions vs. precise ones?

In [ ]:
TEST_SET = [
    ("Flights Agent",    "Which airline has the cheapest layover from Lisbon to Osaka?"),
    ("Flights Agent",    "How many kg of checked baggage are allowed on economy?"),
    ("Visa Agent",       "Does a Brazilian passport need a visa to enter Canada?"),
    ("Visa Agent",       "Do I need a yellow fever vaccination certificate to enter Kenya?"),
    ("Local Tips Agent", "What neighbourhood in Barcelona has the best tapas?"),
    ("Local Tips Agent", "Is it rude to tip in Tokyo?"),
]


def routing_accuracy(route_fn, agents: list, dataset: list) -> float:
    """route_fn(task, agents) -> agent name. Prints each miss and returns the accuracy."""
    hits = 0
    for expected, task in dataset:
        got = route_fn(task, agents)
        if got == expected:
            hits += 1
        else:
            print(f"   ❌ expected {expected!r}, got {got!r}: {task}")
    return hits / len(dataset)


llm_route_name = lambda task, ags: llm_route(task, ags)["agent"]

print("Precise descriptions:")
print("  accuracy =", routing_accuracy(llm_route_name, build_agents(vague=False), TEST_SET))
print("Vague descriptions ('Helps with travel questions.' for all):")
print("  accuracy =", routing_accuracy(llm_route_name, build_agents(vague=True), TEST_SET))

### 🧐 Observation
- With identical vague descriptions, what is the router really guessing from? (Hint: the agent *names*.)
- Good descriptions state **when to use the agent**, list **typical topics**, and (when needed) say **what the agent does NOT handle**. Rewrite one description so it excludes something a neighbour agent handles.

## 5. Routing to Agents That Need Prerequisites 🟡

Some specialists cannot answer from the request alone. A **Trip Budget Agent** needs flight and local price information first.
So *routing can trigger a chain*: after picking the agent, the workflow first runs its **prerequisite agents** and hands their results over as context.

A small **dependency table** keeps this declarative (much easier to extend than nested `if/elif` blocks).

In [ ]:
budget_agent = SpecialistAgent(
    "Trip Budget Agent",
    "Use to estimate the total cost of a trip and split a budget across flights, lodging, food and activities.",
    "You are a trip-budget specialist. Using the provided information, give a budget table with rough numbers and one saving tip. Maximum 120 words.",
)
all_agents = build_agents() + [budget_agent]
by_name = {a.name: a for a in all_agents}

# Which agents must run first, and under which context key their result is passed.
PREREQS = {"Trip Budget Agent": {"flight_info": "Flights Agent", "local_info": "Local Tips Agent"}}


def route_with_prereqs(task: str, agents: list, context: dict | None = None) -> str:
    decision = llm_route(task, agents)
    name = decision["agent"]
    print(f"➡️  Routed to: {name}")
    if name == FALLBACK_NAME:
        return "Sorry, I can't help with that."

    context = dict(context or {})
    for key, prereq_name in PREREQS.get(name, {}).items():
        if key in context:                       # caller already has it, don't pay twice
            continue
        print(f"   ↳ running prerequisite: {prereq_name}")
        context[key] = by_name[prereq_name].run(task)
    return by_name[name].run(task, context)


show("Budget", route_with_prereqs("Roughly how much will a 10-day trip from Madrid to Vietnam cost me?", all_agents))

### 🧐 Observation
- The prerequisites ran **one after the other**. They do not depend on each other. Which pattern from the next notebook would make them faster?
- What could go wrong if `PREREQS` has a cycle (A needs B, B needs A)?

## 6. Embedding-Based Routing (Cosine Similarity) 🔴

Instead of asking an LLM to choose, you can **turn text into vectors** (embeddings) and pick the agent whose description is **closest in meaning** to the request.

1. Embed every agent description **once** and cache the vectors.
2. Embed the incoming request.
3. Compute **cosine similarity** between the request vector and each description vector.
4. Route to the highest score (or fall back if even the best score is low).

$$\cos(a, b) = \frac{a \cdot b}{\lVert a \rVert \, \lVert b \rVert}$$

It is cheaper and more predictable than an LLM call, and you can **log the scores** to see how confident the router was.

In [ ]:
def embed(text: str) -> np.ndarray:
    resp = client.embeddings.create(model=EMBED_MODEL, input=text)
    return np.array(resp.data[0].embedding)


def cosine(a: np.ndarray, b: np.ndarray) -> float:
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))


class EmbeddingRouter:
    def __init__(self, agents: list):
        self.agents = agents
        # computed ONCE, not on every request
        self.description_vectors = {a.name: embed(a.get_description()) for a in agents}

    def scores(self, task: str) -> dict:
        q = embed(task)
        return {name: cosine(q, vec) for name, vec in self.description_vectors.items()}

    def route(self, task: str) -> str:
        scores = self.scores(task)
        return max(scores, key=scores.get)


emb_router = EmbeddingRouter(all_agents)

for q in [
    "Do I need a visa for Canada?",
    "Best street food in Bangkok?",
    "How much will two weeks in Portugal cost?",
    "Write me a poem about autumn.",
]:
    s = emb_router.scores(q)
    best = max(s, key=s.get)
    print(f"{q!r}\n   → {best}   " + "  ".join(f"{k.split()[0]}={v:.2f}" for k, v in s.items()))

### 🧐 Observation
- Look at the scores for the poem request. The router still picks *something*. What does that tell you about needing a **threshold** (fallback when the best score is too low)?
- Cosine similarities between short texts are often in a narrow band (for example 0.1-0.5). Why does that make choosing a threshold an experiment, not a guess?
- Which is better for a call center handling millions of requests: LLM or embedding router? What if requests are long and ambiguous?

### Comparing the two routers on the same test set

In [ ]:
print("LLM router       :", routing_accuracy(lambda t, ags: llm_route(t, ags)["agent"], all_agents, TEST_SET))
print("Embedding router :", routing_accuracy(lambda t, ags: emb_router.route(t), all_agents, TEST_SET))

## 7. Exercises 🏋️

### Exercise 1: A router for a new domain
Build a **fitness coach** with three specialists (Nutrition, Workout Plans, Injury & Recovery) and route requests with `llm_route`.
Write descriptions that are **precise and mutually exclusive**, and write 6 test requests (two per agent) plus one that should hit the fallback.

In [ ]:
fitness_specs = [
    # TODO: (name, description, system_prompt) x 3
]
fitness_agents = [SpecialistAgent(n, d, s) for n, d, s in fitness_specs]

fitness_tests = [
    # TODO: ("Nutrition Agent", "How much protein should I eat on rest days?"), ...
]

# TODO: measure routing_accuracy(llm_route_name, fitness_agents, fitness_tests)

### Exercise 2: Confidence threshold for the embedding router
Subclass `EmbeddingRouter` so `route()` returns `None` when the best cosine score is below a threshold.
1. Print the best score for ~10 on-topic and ~5 off-topic requests to **choose the threshold from data**.
2. Handle `None` with a polite fallback message.

In [ ]:
class ThresholdRouter(EmbeddingRouter):
    def __init__(self, agents: list, threshold: float):
        super().__init__(agents)
        self.threshold = threshold

    def route(self, task: str):
        # TODO: return the best agent name, or None if the best score is below self.threshold
        pass


# TODO: collect scores for on-topic and off-topic requests and pick a threshold that separates them

### Exercise 3: Handle multi-intent requests
"Do I need a visa for Canada and what should I eat in Montreal?" needs **two** agents.
Design (and sketch in code) a router that returns a **list** of agents when a request has several intents, then runs them and merges the answers.
You do not need to make the calls parallel yet, that comes in the next notebook.

In [ ]:
def multi_route(task: str, agents: list) -> list:
    """Return the list of agent names needed for this task (possibly more than one)."""
    # TODO: ask for JSON like {"agents": ["Visa Agent", "Local Tips Agent"]}, validate every name
    pass


def answer_multi(task: str, agents: list) -> str:
    # TODO: run every selected agent and ask one more LLM call to merge their answers
    pass

## 🏁 Summary & Key Takeaways

- A **router** picks *which* specialist handles a request, and runs **only that** one.
- The router only sees **descriptions**: write them as precisely as code. Say what the agent does **and what it does not**.
- Always **validate** the router's answer and have a **fallback**.
- Use **JSON + reason** for logs and debuggability.
- Some agents need **prerequisites**: routing can trigger a small chain (declare the dependencies in a table).
- **Embedding routers** are cheap and inspectable (log the scores, cache the description vectors); **LLM routers** handle nuance. Measure both on a labeled test set.
- One request can have **several intents**: routing alone is not enough, which motivates the next patterns.

### What's next?
- **Deep Dive 4: Parallelization**: run independent agents at the same time and merge their answers.